In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Aesop Fables Corpus Loader

This notebook downloads Project Gutenberg's Aesop collection and parses it into a Python list where each element is one full fable (title + body).

In [1]:
import os

from abstractgraph_generative.story import (
    ask_llm,
    format_text_max_columns,
    load_aesop_fables_from_gutenberg,
    pretty_print_text,
)
from abstractgraph_generative.story.text_hierarchy_hierarchy import (
    hierarchical_story_events as _hierarchical_story_events,
)
from abstractgraph_generative.story.text_hierarchy_render import (
    pretty_print_story_hierarchy,
    print_story_from_hierarchy,
)


In [2]:


def hierarchical_story_events(
    story_text: str,
    model: str | None = None,
    temperature: float = 0.2,
    max_output_tokens: int = 700,
    api_key: str | None = None,
    n_paragraphs: int = 3,
    n_sentences_per_paragraph: int = 3,
    n_events_per_sentence: int = 3,
    enforce_explicit_subjects: bool = True,
    simplify_with_wordnet: bool = False,
    n_abstraction_levels: int = 2,
) -> dict:
    """Notebook-compatible wrapper around modular hierarchy generation.

    Args:
        story_text: Original story text.
        model: Optional model name.
        temperature: Sampling temperature.
        max_output_tokens: Max output tokens per LLM call.
        api_key: Optional API key override.
        n_paragraphs: Number of summary paragraphs.
        n_sentences_per_paragraph: Number of summary sentences per paragraph.
        n_events_per_sentence: Number of atomic events per sentence.
        enforce_explicit_subjects: If True, remove pronouns.
        simplify_with_wordnet: If True, generate abstraction levels.
        n_abstraction_levels: Number of abstraction levels to generate.

    Returns:
        Nested dictionary with paragraphs, sentences, and events.
    """

    return _hierarchical_story_events(
        story_text=story_text,
        ask_llm_fn=ask_llm,
        model=model,
        temperature=temperature,
        max_output_tokens=max_output_tokens,
        api_key=api_key,
        n_paragraphs=n_paragraphs,
        n_sentences_per_paragraph=n_sentences_per_paragraph,
        n_events_per_sentence=n_events_per_sentence,
        enforce_explicit_subjects_flag=enforce_explicit_subjects,
        simplify_with_wordnet=simplify_with_wordnet,
        n_abstraction_levels=n_abstraction_levels,
    )



---

In [3]:
fables = load_aesop_fables_from_gutenberg()
print(f"Parsed {len(fables)} fables.")

Parsed 313 fables.


In [4]:
story_text=fables[5]
pretty_print_text(text=story_text, max_columns=88)
print('-'*100)

n_splits = 3
hierarchy = hierarchical_story_events(
    story_text=story_text,
    model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
    temperature=0.2,
    max_output_tokens=900,
    n_paragraphs=n_splits,
    n_sentences_per_paragraph=n_splits-1,
    n_events_per_sentence=n_splits-1,
    simplify_with_wordnet=True,
    n_abstraction_levels=3,
)
pretty_print_story_hierarchy(hierarchy)

The Bat And The Weasels

A BAT who fell upon the ground and was caught by a Weasel pleaded to be spared his life.
The Weasel refused, saying that he was by nature the enemy of all birds. The Bat assured
him that he was not a bird, but a mouse, and thus was set free. Shortly afterwards the
Bat again fell to the ground and was caught by another Weasel, whom he likewise
entreated not to eat him. The Weasel said that he had a special hostility to mice. The
Bat assured him that he was not a mouse, but a bat, and thus a second time escaped.
----------------------------------------------------------------------------------------------------
Hierarchy config:
- paragraphs: 3
- sentences/paragraph: 2
- events/sentence: 2
- simplify with wordnet: True
- n_abstraction_levels: 3

Paragraph 1:
A BAT fell to the ground and was captured by a Weasel, who refused to spare the BAT's life due to a natural enmity towards birds.

  Sentence 1.1:
  A BAT fell to the ground and was captured by a Weasel.
  Ev

In [5]:
pretty_print_text(text=story_text, max_columns=88)

print('-'*100)
print_story_from_hierarchy(hierarchy, level="paragraph")
print('-'*100)
print_story_from_hierarchy(hierarchy, level="sentence")
print('-'*100)
print_story_from_hierarchy(hierarchy, level="event")
print('-'*100)
print_story_from_hierarchy(hierarchy, level="simplified", abstraction_level=1)
print('-'*100)
print_story_from_hierarchy(hierarchy, level="simplified", abstraction_level=2)
print('-'*100)
    

The Bat And The Weasels

A BAT who fell upon the ground and was caught by a Weasel pleaded to be spared his life.
The Weasel refused, saying that he was by nature the enemy of all birds. The Bat assured
him that he was not a bird, but a mouse, and thus was set free. Shortly afterwards the
Bat again fell to the ground and was caught by another Weasel, whom he likewise
entreated not to eat him. The Weasel said that he had a special hostility to mice. The
Bat assured him that he was not a mouse, but a bat, and thus a second time escaped.
----------------------------------------------------------------------------------------------------
level: paragraph
A BAT fell to the ground and was captured by a Weasel, who refused to spare the BAT's
life due to a natural enmity towards birds.

The BAT claimed to be a mouse, convincing the Weasel to release the BAT.

After falling again, the BAT encountered another Weasel, who expressed hostility towards
mice, prompting the BAT to assert his identity 